# T66-负面舆情爬取

本Notebook实现负面舆情爬取功能，包括：
- 企业预警通API登录认证
- 舆情数据获取
- OCR图像识别
- 大模型内容总结
- 敏感词过滤
- 数据库存储

## 功能说明

1. **API认证**：登录Ratingdog获取访问令牌
2. **舆情采集**：获取指定日期范围的负面舆情
3. **内容处理**：OCR识别+敏感词过滤+AI总结
4. **数据存储**：将结果存储到MySQL数据库

## 1. 环境配置与依赖导入

导入所需的库并加载配置文件。

In [ ]:
# 环境配置与依赖导入
import os
import sys
import re
import io
import json
import base64
import logging
from datetime import datetime, timedelta
from pathlib import Path

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text

# 图像处理
from PIL import Image
import cv2

# 网页解析
from bs4 import BeautifulSoup

# HTTP请求
import requests

# OpenAI客户端
import openai

# 敏感词过滤
import ahocorasick

# OCR
try:
    from paddleocr import PaddleOCR
    PADDLE_AVAILABLE = True
except ImportError:
    PADDLE_AVAILABLE = False

# 导入配置
from config import (
    get_db_connection_string,
    RATINGDOG_LOGIN_URL,
    RATINGDOG_API_URL,
    RATINGDOG_USERNAME,
    RATINGDOG_PHONE_CODE,
    RATINGDOG_PASSWORD,
    RATINGDOG_TENANT_ID,
    REQUEST_TIMEOUT,
    DEFAULT_PAGE_SIZE,
    CODE_LIBRARY_PATH
)

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 设置环境变量
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

print("依赖导入完成")

## 2. 数据库与OCR配置

配置数据库连接和OCR引擎。

In [ ]:
# 数据库与OCR配置

# 创建数据库引擎
db_engine = sqlalchemy.create_engine(
    get_db_connection_string(),
    poolclass=sqlalchemy.pool.NullPool,
    pool_recycle=3600
)
print("数据库连接创建成功")

# 初始化OCR引擎
if PADDLE_AVAILABLE:
    try:
        ocr = PaddleOCR(
            use_angle_clsf=True,
            lang='ch',
            show_log=False,
            use_gpu=True
        )
        print("PaddleOCR初始化成功")
    except Exception as e:
        logging.warning(f"PaddleOCR初始化失败: {e}")
        ocr = None
else:
    ocr = None
    print("PaddleOCR不可用")

## 3. 敏感词过滤

实现敏感词加载和过滤功能。

In [ ]:
# 敏感词过滤

# 默认敏感词列表（可根据需要从文件加载）
DEFAULT_SENSITIVE_WORDS = []

def build_automaton(sensitive_words):
    """构建Aho-Corasick自动机
    
    Args:
        sensitive_words: 敏感词列表
        
    Returns:
        Automaton: Aho-Corasick自动机
    """
    A = ahocorasick.Automaton()
    for idx, word in enumerate(sensitive_words):
        if word:
            A.add_word(word, (idx, word))
    A.make_automaton()
    return A

def filter_sensitive_words(text, automaton):
    """过滤文本中的敏感词
    
    Args:
        text: 待过滤文本
        automaton: Aho-Corasick自动机
        
    Returns:
        str: 过滤后的文本
    """
    if not automaton:
        return text
        
    matches = []
    for end_index, (insert_order, original_value) in automaton.iter(text):
        start_index = end_index - len(original_value) + 1
        matches.append((start_index, end_index + 1))

    filtered_text_parts = []
    last_end = 0
    for start, end in matches:
        filtered_text_parts.append(text[last_end:start])
        last_end = end
    filtered_text_parts.append(text[last_end:])

    return ''.join(filtered_text_parts)

# 构建自动机
automaton = build_automaton(DEFAULT_SENSITIVE_WORDS)
print("敏感词过滤模块初始化完成")

## 4. OCR与内容处理

实现OCR识别和HTML内容处理功能。

In [ ]:
# OCR与内容处理

def recognize_image(img_array):
    """OCR识别图像
    
    Args:
        img_array: 图像数组
        
    Returns:
        str: 识别的文本
    """
    if ocr is None:
        return ''
        
    try:
        results = ocr.ocr(img_array, cls=True)
        result = results[0] if results else []
        text = ''
        for item in result:
            try:
                text += item[1][0] + '\n'
            except Exception:
                pass
        return text
    except Exception as e:
        logging.error(f"OCR识别失败: {e}")
        return ''

def preprocess_image(img_array):
    """图像预处理
    
    Args:
        img_array: 图像数组
        
    Returns:
        str: 识别的文本
    """
    try:
        gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU
        )
        _, dist = cv2.threshold(
            thresh, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU
        )
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 1))
        opening = cv2.morphologyEx(dist, cv2.MORPH_OPEN, kernel)
        return recognize_image(opening)
    except Exception as e:
        logging.error(f"图像预处理失败: {e}")
        return ''

def download_image(url):
    """下载网络图片
    
    Args:
        url: 图片URL
        
    Returns:
        PIL.Image: 图片对象
    """
    try:
        response = requests.get(url, stream=True, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        image_stream = io.BytesIO(response.content)
        return Image.open(image_stream)
    except Exception as e:
        logging.error(f"图片下载失败: {e}")
        return None

def handle_base64_image(base64_string):
    """处理Base64编码的图片
    
    Args:
        base64_string: Base64编码的图片字符串
        
    Returns:
        PIL.Image: 图片对象
    """
    try:
        image_data = re.sub('^data:image/.+;base64,', '', base64_string)
        byte_data = base64.b64decode(image_data)
        return Image.open(io.BytesIO(byte_data))
    except Exception as e:
        logging.error(f"Base64图片处理失败: {e}")
        return None

def get_html_text_and_images(html_content):
    """提取HTML文本和图像
    
    Args:
        html_content: HTML内容
        
    Returns:
        tuple: (文本, 图像文本列表)
    """
    content = ''
    image_texts = []
    image_counter = 1
    
    try:
        soup = BeautifulSoup(html_content, 'html.parser')
        content = soup.get_text()
        images = soup.find_all(['img', 'picture'])
        
        for img in images:
            try:
                img_src = None
                if img.name == 'img':
                    img_src = img.get('data-src') or img.get('src')
                elif img.name == 'picture':
                    img_tag = img.find('img')
                    if img_tag:
                        img_src = img_tag.get('src')
                        
                if not img_src:
                    continue
                    
                if img_src.startswith('data:'):
                    image = handle_base64_image(img_src)
                elif img_src.startswith(('http://', 'https://')):
                    image = download_image(img_src)
                else:
                    continue
                    
                if image:
                    ocr_result = preprocess_image(np.array(image))
                    image_texts.append((f"---IMAGE {image_counter}---", ocr_result))
                    image_counter += 1
                    
            except Exception as e:
                logging.error(f"图像处理失败: {e}")
                
    except Exception as e:
        logging.error(f"HTML解析失败: {e}")
        
    return content, image_texts

print("OCR与内容处理函数定义完成")

## 5. Ratingdog API客户端

实现Ratingdog API认证和数据获取功能。

In [ ]:
# Ratingdog API客户端

class RatingdogClient:
    """Ratingdog API客户端"""
    
    def __init__(self):
        """初始化客户端"""
        self.access_token = None
        self.headers = None
        
    def login(self):
        """登录获取访问令牌
        
        Returns:
            bool: 登录是否成功
        """
        login_headers = {
            'Accept': 'application/json, text/plain, */*',
            'Content-Type': 'application/json;charset=UTF-8',
            'Devicechannel': 'gclife_bmp_pc',
            'Origin': 'https://www.ratingdog.cn',
            'Ratingdog.Tenantid': RATINGDOG_TENANT_ID,
            'Referer': 'https://www.ratingdog.cn/',
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        login_data = {
            'UserNameOrEmailAddressOrPhone': RATINGDOG_USERNAME,
            'internationalPhoneCode': RATINGDOG_PHONE_CODE,
            'password': RATINGDOG_PASSWORD
        }
        
        try:
            response = requests.post(
                RATINGDOG_LOGIN_URL,
                headers=login_headers,
                json=login_data,
                timeout=REQUEST_TIMEOUT
            )
            
            result = response.json()
            self.access_token = result['result']['accessToken']
            
            # 设置API请求头
            self.headers = {
                'Accept': 'application/json, text/plain, */*',
                'Content-Type': 'application/json; charset=UTF-8',
                'Authorization': f'Bearer {self.access_token}',
                'Devicechannel': 'gclife_bmp_pc',
                'Origin': 'https://www.ratingdog.cn',
                'Ratingdog.Tenantid': RATINGDOG_TENANT_ID,
                'Referer': 'https://www.ratingdog.cn/',
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            
            logging.info("Ratingdog登录成功")
            return True
            
        except Exception as e:
            logging.error(f"Ratingdog登录失败: {e}")
            return False
            
    def get_opinion_detail(self, opinion_id):
        """获取舆情详情
        
        Args:
            opinion_id: 舆情ID
            
        Returns:
            dict: 舆情详情
        """
        url = f"{RATINGDOG_API_URL}/GetCiPublicOpinionForView?id={opinion_id}"
        
        try:
            response = requests.get(
                url,
                headers=self.headers,
                timeout=REQUEST_TIMEOUT
            )
            result = response.json()
            return result.get('result')
        except Exception as e:
            logging.error(f"获取舆情详情失败 [{opinion_id}]: {e}")
            return None
            
    def get_opinions_by_date(self, start_date, end_date, skip_count=0):
        """按日期获取舆情列表
        
        Args:
            start_date: 开始日期（YYYY-MM-DD）
            end_date: 结束日期（YYYY-MM-DD）
            skip_count: 跳过记录数
            
        Returns:
            dict: 舆情列表结果
        """
        url = f"{RATINGDOG_API_URL}/GetQueryPublicOpinionsForTenant"
        
        data = {
            'IsRisk': True,
            'MoodLevels': [],
            'StartDate': start_date,
            'EndDate': end_date,
            'Type': '',
            'administrativeDivisionIds': [],
            'industryIds': [],
            'Source': False,
            'MaxResultCount': DEFAULT_PAGE_SIZE,
            'SkipCount': skip_count
        }
        
        try:
            response = requests.post(
                url,
                headers=self.headers,
                json=data,
                timeout=REQUEST_TIMEOUT
            )
            return response.json()
        except Exception as e:
            logging.error(f"获取舆情列表失败: {e}")
            return {'result': {'totalCount': 0, 'items': []}}

print("RatingdogClient类定义完成")

## 6. 数据处理与存储

实现舆情数据处理和存储功能。

In [ ]:
# 数据处理与存储

def parse_opinion_detail(detail):
    """解析舆情详情
    
    Args:
        detail: 舆情详情字典
        
    Returns:
        dict: 解析后的数据
    """
    if detail is None:
        return None
        
    # 提取城市信息
    city_names = detail.get('cityFullNames', []) or []
    city_full_names = ','.join([c for c in city_names if c])
    
    # 提取行业信息
    industry_names = detail.get('industryFullNames', []) or []
    industry_full_names = ','.join([i for i in industry_names if i])
    
    # 提取关键词公司
    keywords = detail.get('miniIssuers', []) or []
    list_keywords = ','.join([k.get('issuerName', '') for k in keywords if k.get('issuerName')])
    
    # 提取发生日期
    happen_date = detail.get('happenDate', '')
    
    # 提取信息类型
    type_info_part1 = detail.get('parentPublicOpinionTypeName', '') or ''
    type_info_part2 = detail.get('publicOpinionTypeName', '') or ''
    type_info = type_info_part1 + type_info_part2
    
    # 提取内容
    content, image_texts = get_html_text_and_images(
        html_content=detail.get('publicOpinionContent', '')
    )
    
    commentary = detail.get('commentary', '') or ''
    content_info = commentary + (content or '') + str(image_texts)
    
    return {
        'list_keywords': list_keywords,
        'type_info': type_info,
        'happen_date': happen_date,
        'content_info': content_info,
        'city_full_names': city_full_names,
        'industry_full_names': industry_full_names
    }

def summarize_content_with_ai(content, type_info):
    """使用AI总结内容
    
    Args:
        content: 待总结内容
        type_info: 类型信息
        
    Returns:
        str: 总结内容
    """
    # 这里可以集成大模型API进行总结
    # 目前返回简单的截取
    if len(content) > 500:
        return content[:500] + '...'
    return content

def insert_opinion_to_db(conn, data):
    """插入舆情数据到数据库
    
    Args:
        conn: 数据库连接
        data: 舆情数据字典
    """
    sql = """
    INSERT IGNORE INTO 舆情监控 
    (list_keywords, TypeInfo, happenDate, ContentInfo, ai_sum, cityFullNames, industryFullNames)
    VALUES (:list_keywords, :TypeInfo, :happenDate, :ContentInfo, :ai_sum, :cityFullNames, :industryFullNames)
    """
    
    params = {
        'list_keywords': data['list_keywords'],
        'TypeInfo': data['type_info'],
        'happenDate': data['happen_date'],
        'ContentInfo': data['content_info'],
        'ai_sum': data['ai_sum'],
        'cityFullNames': data['city_full_names'],
        'industryFullNames': data['industry_full_names']
    }
    
    conn.execute(text(sql), params)

print("数据处理与存储函数定义完成")

## 7. 主程序执行

执行完整的舆情爬取流程。

In [ ]:
# 主程序执行

def collect_opinions(client, start_date, end_date):
    """采集指定日期范围的舆情
    
    Args:
        client: Ratingdog客户端
        start_date: 开始日期
        end_date: 结束日期
    """
    current_date = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    while current_date <= end:
        date_str = current_date.strftime('%Y-%m-%d')
        print(f"\n处理日期: {date_str}")
        
        skip_count = 0
        is_end = False
        
        while not is_end:
            result = client.get_opinions_by_date(date_str, date_str, skip_count)
            total_count = result['result']['totalCount']
            
            if total_count > 0:
                for item in result['result']['items']:
                    opinion_id = item['id']
                    detail = client.get_opinion_detail(opinion_id)
                    
                    if detail is None:
                        continue
                        
                    parsed_data = parse_opinion_detail(detail)
                    if parsed_data is None:
                        continue
                        
                    # 过滤敏感词
                    content_text = (
                        f"涉及公司：{parsed_data['list_keywords']}\n"
                        f"负面信息类型：{parsed_data['type_info']}\n"
                        f"事件发生日期：{parsed_data['happen_date']}\n"
                        f"具体负面信息内容：{parsed_data['content_info']}"
                    )
                    filtered_content = filter_sensitive_words(content_text, automaton)
                    
                    # AI总结
                    ai_sum = summarize_content_with_ai(
                        filtered_content,
                        parsed_data['type_info']
                    )
                    parsed_data['ai_sum'] = ai_sum
                    
                    # 存储到数据库
                    try:
                        with db_engine.begin() as conn:
                            insert_opinion_to_db(conn, parsed_data)
                        print(f"  舆情保存成功: {opinion_id}")
                    except Exception as e:
                        logging.error(f"  舆情保存失败 [{opinion_id}]: {e}")
                        
            # 检查是否还有更多数据
            if skip_count + DEFAULT_PAGE_SIZE >= total_count:
                is_end = True
            else:
                skip_count += DEFAULT_PAGE_SIZE
                
        current_date += timedelta(days=1)

def main():
    """主执行函数"""
    print("=" * 50)
    print("T66-负面舆情爬取 任务开始")
    print(f"执行时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 50)
    
    # 1. 登录Ratingdog
    print("\n[1/2] 登录Ratingdog...")
    client = RatingdogClient()
    if not client.login():
        print("登录失败，任务终止")
        return
        
    # 2. 采集舆情
    print("\n[2/2] 采集舆情数据...")
    # 设置日期范围（示例：最近7天）
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
    
    collect_opinions(client, start_date, end_date)
        
    print("\n" + "=" * 50)
    print("T66-负面舆情爬取 任务完成")
    print("=" * 50)

# 执行主程序
if __name__ == '__main__':
    main()

## 总结

本Notebook实现了以下功能：

1. **API认证**：登录Ratingdog获取访问令牌
2. **舆情采集**：按日期范围获取负面舆情列表
3. **详情获取**：获取每条舆情的详细信息
4. **OCR识别**：识别舆情内容中的图像文字
5. **敏感词过滤**：过滤敏感词内容
6. **数据存储**：将结果存储到MySQL数据库

### 使用说明

1. 确保已安装所有依赖包：`pip install -r requirements.txt`
2. 配置环境变量或修改config.py中的默认值
3. 确保有Ratingdog账号和访问权限
4. 运行主程序执行完整流程
5. 可通过修改日期范围控制采集范围